# 🏡 Interior Design Generator using GenAI

This project uses Stable Diffusion (SDXL Turbo) to generate interior design images from input images.

It allows users to transform room designs using AI with a simple interface.

## 🚀 Features

- Image-to-image transformation
- Uses SDXL Turbo (fast generation)
- Simple Gradio interface
- GPU accelerated

In [ ]:
!pip install -q diffusers transformers accelerate safetensors gradio

## 📦 Import Libraries



In [ ]:

import torch
from diffusers import AutoPipelineForImage2Image
from PIL import Image
import gradio as gr
import random


## ⚙️ Device Setup


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device != "cuda":
    raise RuntimeError("❌ Please enable GPU in Colab (Runtime → Change runtime → GPU)")


## 🤖 Load Model (SDXL Turbo)

In [ ]:

print("⏳ Loading model...")

pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

pipe.enable_attention_slicing()

print("✅ Model Loaded")

## 🎨Style Prompt

In [ ]:
STYLE_PROMPTS = {
    "Modern Luxury": "ultra realistic modern luxury bedroom, premium furniture, elegant lighting, high-end interior design, photorealistic, 8k",
    "Minimalist": "minimalist bedroom, clean white tones, simple furniture, natural lighting, modern interior, photorealistic",
    "Bohemian": "bohemian bedroom, plants, warm cozy lighting, artistic decor, textured fabrics, photorealistic",
    "Japanese Zen": "japanese zen room, wooden textures, peaceful atmosphere, natural light, minimal decor, photorealistic",
    "Industrial": "industrial loft bedroom, concrete walls, metal furniture, dramatic lighting, modern industrial design"
}

NEGATIVE_PROMPT = "blurry, distorted, unrealistic, cartoon, low quality, bad perspective"

## Generation Function

In [ ]:

def generate_design(image, style, creativity):
    image = image.resize((768, 768))

    prompt = f"""
    {STYLE_PROMPTS[style]},
    highly detailed, sharp focus, realistic textures, DSLR photo
    """

    outputs = []

    for i in range(3):
        seed = random.randint(0, 100000)
        generator = torch.manual_seed(seed)

        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=image,
            strength=float(creativity),
            guidance_scale=7.5,
            num_inference_steps=30,
            generator=generator
        ).images[0]

        outputs.append(result)

    return outputs

## 🌐 Gradio Interface





In [ ]:

with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🏠 AI Interior Designer Pro (Improved Version)")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(type="pil", label="Upload Room Image")

            style = gr.Dropdown(
                choices=list(STYLE_PROMPTS.keys()),
                value="Modern Luxury",
                label="Select Style"
            )

            creativity = gr.Slider(
                minimum=0.2,
                maximum=0.9,
                value=0.5,
                step=0.1,
                label="Creativity Level (Low = Small change, High = Big change)"
            )

            generate_btn = gr.Button("✨ Generate Designs")

        with gr.Column():
            output_gallery = gr.Gallery(label="Generated Designs", columns=3)

    generate_btn.click(
        fn=generate_design,
        inputs=[input_image, style, creativity],
        outputs=output_gallery
    )



## ▶️ Run the App

In [ ]:
app.launch(share=True)